# Task 4 — Prediction / Forecast Script

This notebook brings Tasks 1–3 together into one pipeline that forecasts hourly westbound traffic volume on I-94:

| Step | What happens | Code used |
| :-- | :-- | :-- |
| 1. Fetch | Get the latest record and the 14 days before it **from the Task 3 API** (MySQL and MongoDB) | `src/api_client.py` |
| 2. Preprocess | Apply the **same Task 1 pipeline**: cleaning, hourly grid, gap handling, lags, moving averages | `src/preprocessing.py` |
| 3. Load model | Load the tuned Random Forest saved in Task 1 (experiment 4) | `models/best_model.joblib` |
| 4. Predict | Forecast the next hour and the next 24 hours, then check accuracy against real values | `src/forecasting.py` |

The same pipeline is also available as a command-line script, `scripts/predict.py`, which the last section runs to save the output files.

**How to run:** in Google Colab, choose *Runtime → Run all*. The setup section installs MySQL, MongoDB and the API dependencies inside Colab, loads the cleaned data into both databases, and starts the API. Colab cannot run Docker, so this replaces `docker compose up`. On a laptop where the API is already running (see README), the setup section skips itself.

## 0. Setup

In [ ]:
import os, sys, time, json, socket, subprocess
from pathlib import Path

try:                                    # Google Colab: project folder lives on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                     # local Jupyter: this notebook sits in <project>/notebooks
    IN_COLAB = False

PROJECT = Path("/content/drive/MyDrive/traffic-project") if IN_COLAB else Path.cwd().parent
PROJECT = Path(os.environ.get("TRAFFIC_PROJECT", PROJECT))
API_URL = "http://127.0.0.1:8000"
START_LOCAL_STACK = IN_COLAB            # Colab: install MySQL + MongoDB and start the API here

assert (PROJECT / "scripts" / "predict.py").exists(), f"Project folder not found at {PROJECT}"
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project folder:", PROJECT, "| Colab:", IN_COLAB)

def run(cmd, check=True, quiet=False):
    """Run a shell command and show the tail of its output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet or r.returncode:
        print((r.stdout + r.stderr)[-1500:])
    if check and r.returncode:
        raise RuntimeError(f"Command failed: {cmd}")
    return r.returncode == 0

def port_open(port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

def wait_for(port, seconds=60):
    for _ in range(seconds):
        if port_open(port):
            return True
        time.sleep(1)
    return False

### 0.1 Install the API dependencies and MySQL

The credentials match `docker-compose.yml` (user `root`, password `root`, database `traffic_db`), so the Task 3 API runs without changes.

In [ ]:
if START_LOCAL_STACK:
    run(f"{sys.executable} -m pip install -q fastapi uvicorn pymysql pymongo python-dotenv", quiet=True)
    if not port_open(3306):
        run("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null",
            quiet=True)
        run("service mysql start", check=False, quiet=True)
        wait_for(3306)
    setup_sql = ("CREATE DATABASE IF NOT EXISTS traffic_db; "
                 "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root'; "
                 "CREATE USER IF NOT EXISTS 'root'@'127.0.0.1' IDENTIFIED WITH mysql_native_password BY 'root'; "
                 "GRANT ALL PRIVILEGES ON *.* TO 'root'@'127.0.0.1' WITH GRANT OPTION; FLUSH PRIVILEGES;")
    ok = (run(f'mysql -uroot -e "{setup_sql}"', check=False, quiet=True)
          or run(f'mysql -uroot -proot -e "{setup_sql}"', check=False, quiet=True))   # re-run: password already set
    print("MySQL ready" if ok and port_open(3306) else "MySQL setup FAILED - see output above")
else:
    print("Skipped: using the API you started yourself at", API_URL)

### 0.2 Install and start MongoDB

In [ ]:
MONGO_HOME = Path("/content/mongodb")
MONGO_DOWNLOADS = [   # Colab runs Ubuntu 22.04
    "https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-7.0.14.tgz",
    "https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-8.0.12.tgz",
]
if START_LOCAL_STACK and not port_open(27017):
    mongod = next(MONGO_HOME.glob("*/bin/mongod"), None)
    for url in MONGO_DOWNLOADS:
        if mongod:
            break
        run(f"mkdir -p {MONGO_HOME} && curl -fsSL {url} | tar -xz -C {MONGO_HOME}", check=False, quiet=True)
        mongod = next(MONGO_HOME.glob("*/bin/mongod"), None)
    if mongod:
        run(f"mkdir -p /content/mongo-data && {mongod} --dbpath /content/mongo-data --bind_ip 127.0.0.1 "
            "--port 27017 --fork --logpath /content/mongod.log", check=False, quiet=True)
        wait_for(27017, 30)
if START_LOCAL_STACK:
    print("MongoDB ready" if port_open(27017) else "MongoDB not running - the notebook will continue with MySQL only")
else:
    print("Skipped: using the API you started yourself at", API_URL)

### 0.3 Load the cleaned data into both databases and start the Task 3 API

`scripts/seed_databases.py` (Task 3) loads `data/traffic_clean.csv` into the four MySQL tables and the MongoDB `traffic` collection. Loading takes 1–3 minutes.

In [ ]:
if START_LOCAL_STACK and not port_open(8000):
    run(f"{sys.executable} scripts/seed_databases.py", check=False)
    api_log = open("/content/api.log", "w")
    API_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd=PROJECT, stdout=api_log, stderr=subprocess.STDOUT)
    print("API started" if wait_for(8000) else "API did not start - see /content/api.log")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    display
except NameError:
    display = print

from src.api_client import TrafficApiClient
from src.preprocessing import add_features
from src.forecasting import load_model, preprocess, one_step_backtest, recursive_forecast, metrics

client = TrafficApiClient(API_URL)
health = client.health()
print("API health:", health)
SOURCES = [s for s, key in [("sql", "mysql"), ("mongo", "mongodb")] if health.get(key) == "ok"]
NAMES = {"sql": "MySQL", "mongo": "MongoDB"}
assert SOURCES, "Neither database is reachable through the API"
print("Databases used:", [NAMES[s] for s in SOURCES])

## 1. Fetch time-series records from the API

Two Task 3 endpoints are used:

- `GET /{sql|mongo}/observations/latest` returns the newest stored hour.
- `GET /{sql|mongo}/observations/range?start=…&end=…` returns the 336 hours (14 days) up to and including that hour.

The model's longest features (`lag_168`, `ma_168`) need the previous full week, so at least 168 hours of history are required. 14 days leaves room for gaps and for the accuracy check.

In [ ]:
HISTORY_HOURS = 336
fetched = {}
for src in SOURCES:
    latest = client.latest(src)
    end = pd.Timestamp(latest["date_time"]).to_pydatetime()
    start = end - pd.Timedelta(hours=HISTORY_HOURS - 1)
    records = client.date_range(src, start, end)
    fetched[src] = records
    print(f"{NAMES[src]:8s} latest = {latest['date_time']} (traffic_volume={latest['traffic_volume']}), "
          f"range {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M}: {len(records)} records")

print("\nExample record as returned by the API:")
print(json.dumps(fetched[SOURCES[0]][-1], indent=2))

In [ ]:
pd.json_normalize(fetched[SOURCES[0]]).tail()

## 2. Preprocess with the Task 1 pipeline

The API records are turned back into the raw dataset's columns. The **same functions used to train the model** are then applied, imported from `src/preprocessing.py`, so nothing is re-implemented:

- `clean_traffic_data()` parses timestamps, removes sensor errors (temperature ≤ 0 K, rain > 500 mm), puts the data on a strict hourly grid, interpolates gaps of up to 6 hours and drops longer gaps.
- `add_features()` builds calendar features (`hour`, `dayofweek`, `month`, `is_holiday`), lags (`lag_1`, `lag_24`, `lag_168`) and moving averages of past values (`ma_24`, `ma_168`).

In [ ]:
clean = {src: preprocess(fetched[src]) for src in SOURCES}
c = clean[SOURCES[0]]
print(f"{len(fetched[SOURCES[0]])} API records -> {len(c)} clean hourly rows, "
      f"{int(c['is_imputed'].sum())} interpolated, range {c.index.min()} -> {c.index.max()}")

features_df = add_features(c)
features_df[["traffic_volume", "temp_k", "hour", "dayofweek", "is_holiday",
             "lag_1", "lag_24", "lag_168", "ma_24", "ma_168"]].tail(6)

**Check that the API pipeline matches training.** Features built from the API records are compared with features built from the training CSV (`data/traffic_clean.csv`) for the same hours. Identical values mean the model receives exactly the inputs it was trained on.

In [ ]:
train_df = pd.read_csv(PROJECT / "data" / "traffic_clean.csv", parse_dates=["timestamp"], index_col="timestamp")
train_df["holiday"] = train_df["holiday"].fillna("None")
cols = ["lag_1", "lag_24", "lag_168", "ma_24", "ma_168", "hour", "dayofweek", "temp_k"]
last_day = c.index[-24:]
same = np.allclose(add_features(train_df).loc[last_day, cols].to_numpy(dtype=float),
                   features_df.loc[last_day, cols].to_numpy(dtype=float))
print("API features identical to training features for the last 24 hours:", same)

## 3. Load the trained model

The model is the tuned Random Forest from Task 1 experiment 4 (`max_depth=12`, `max_features=0.5`, `n_estimators=100`), which had the best validation RMSE. `load_model()` also checks that the model's feature order matches `models/features.json`.

In [ ]:
model, FEATURES, TARGET, notes = load_model(PROJECT / "models")
for n in notes:
    print("Note:", n)
print(model)
print("Target:", TARGET)
print("Features:", FEATURES)
display(pd.read_csv(PROJECT / "models" / "experiment_table.csv"))

## 4. Make the prediction / forecast

### 4.1 Next-hour and 24-hour forecast

The next hour is predicted from observed values only. For the following hours, each prediction is fed back in as `lag_1` and into the moving averages (a *recursive* forecast). Calendar features are known exactly. Future weather is not stored anywhere, so it is carried forward from the last observed hour. Weather only weakly affects traffic (Task 1, Q5), so this assumption costs little accuracy.

In [ ]:
HORIZON = 24
forecasts = {src: recursive_forecast(clean[src], model, FEATURES, HORIZON) for src in SOURCES}
fc = forecasts[SOURCES[0]]
nxt = fc.iloc[0]
print(f">>> Next-hour forecast for {fc.index[0]:%A %Y-%m-%d %H:%M}: {int(nxt['predicted'])} vehicles/hour <<<")
if len(SOURCES) == 2:
    same = (forecasts["sql"]["predicted"] == forecasts["mongo"]["predicted"]).all()
    print("MySQL and MongoDB forecasts identical:", same)

table = fc[["predicted"]].copy()
table.insert(0, "day", table.index.day_name())
table.T

In [ ]:
recent = clean[SOURCES[0]]["traffic_volume"].iloc[-72:]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(recent.index, recent, color="#333", label="Observed (fetched from API)")
ax.plot(fc.index, fc["predicted"], color="#d1495b", marker="o", ms=3, lw=2, label=f"Forecast, next {HORIZON} h")
ax.axvline(recent.index[-1], color="grey", ls=":")
ax.set_ylabel("vehicles per hour"); ax.set_title("Task 4: traffic volume forecast after the latest record")
ax.grid(alpha=.3); ax.legend(); plt.show()

### 4.2 How accurate is it? Checks against real values

1. **One-step check:** each of the last 24 fetched hours is predicted from its real lags and compared with what happened and with the seasonal-naive baseline (same hour last week, `lag_168`).
2. **Past-day forecast:** the forecast is started one week earlier (Sunday 2018-09-23 23:00), using only data up to that point fetched from the API. Its 24 predictions are then compared with the actual Monday values, which the API also returns.

In [ ]:
bt = one_step_backtest(clean[SOURCES[0]], model, FEATURES, 24)
m_model = metrics(bt["actual"], bt["predicted"])
m_base = metrics(bt["actual"], bt["baseline_lag168"])
print("One-step, last 24 h   model:", m_model, "| lag_168 baseline:", m_base)

AS_OF = pd.Timestamp("2018-09-23 23:00:00")
src = SOURCES[0]
hist = client.date_range(src, (AS_OF - pd.Timedelta(hours=HISTORY_HOURS - 1)).to_pydatetime(), AS_OF.to_pydatetime())
past_fc = recursive_forecast(preprocess(hist), model, FEATURES, 24)
actual = client.date_range(src, past_fc.index[0].to_pydatetime(), past_fc.index[-1].to_pydatetime())
past_fc["actual"] = pd.Series({pd.Timestamp(r["date_time"]): r["traffic_volume"] for r in actual})
print("24-h forecast from", AS_OF, "vs actual:", metrics(past_fc["actual"], past_fc["predicted"]))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(past_fc.index, past_fc["actual"], color="#333", label="Actual (from API)")
ax.plot(past_fc.index, past_fc["predicted"], color="#d1495b", marker="o", ms=3, label="Forecast")
ax.set_title("Forecast for Monday 2018-09-24 made with data up to Sunday 23:00")
ax.set_ylabel("vehicles per hour"); ax.grid(alpha=.3); ax.legend(); plt.show()

## 5. Run the command-line script and save the results

`scripts/predict.py` runs the same four steps from the terminal. It saves the outputs used in the report:

- `results/task4_forecast_{sql,mongo}.csv|json` and `figures/12_forecast_{sql,mongo}.png`: forecast from the latest record
- `results/task4_backtest_sql_20180923_23.csv|json` and `figures/13_forecast_vs_actual_sql.png`: past-day check against actual values

In [ ]:
source = "both" if len(SOURCES) == 2 else SOURCES[0]
run(f"{sys.executable} scripts/predict.py --source {source}")
run(f'{sys.executable} scripts/predict.py --source {SOURCES[0]} --as-of "2018-09-23 23:00:00"')

In [ ]:
run(f"{sys.executable} -m pytest -q tests", check=False)    # offline tests for the Task 4 pipeline

## 6. Interpretation

- **The pipeline runs end to end.** Data comes only from the API, is processed by the same Task 1 code used in training (the feature check above prints `True`), and the saved model produces the forecast. With both databases loaded from the same cleaned CSV, MySQL and MongoDB give identical forecasts.
- **The forecast is realistic.** For Monday 2018-10-01 it shows the weekday pattern found in Task 1 (Q2): night-time traffic of about 300 vehicles/hour, a morning peak around 07:00 and an evening peak around 16:00, both above 6,000.
- **The accuracy is close to the Task 1 test results.** The one-step error on the latest day (MAE ≈ 119) is well below the same-hour-last-week baseline (≈ 224). The 24-hour forecast made a week earlier has MAE ≈ 153, equal to the model's test MAE, so feeding predictions back in does not make errors grow quickly within a day.
- **Limitations.** Future weather is assumed to stay the same as the last observed hour. Public holidays are only known if they already appear in the fetched history. A real deployment would add a weather forecast and a holiday calendar as inputs.